# DE4 — Sparse customer × product matrix, and DE5 — matrix report

**Project:** DS 5020 Mini-Project "Customers Like You" (PCA/SVD recommender on UCI Online Retail II).

Input: `data/processed/train.parquet` **only**. The matrix is built from training data; the test targets are read
*after* the matrix exists and are used only to flag which targets are reachable (step 7).
Logic lives in `src/build_matrix.py` (`build_matrix(train_df, value="log", min_customers=10)`), unit-tested in
`tests/test_build_matrix.py`.

Steps (fixed order):

1. **Aggregate** `Quantity` per `(Customer ID, StockCode)` → `total_qty`, one row per pair — *before* any transform.
   Applying `log1p` per invoice line and letting `csr_matrix` sum duplicates would give
   \(\ln 13 + \ln 13 = \ln 169\) instead of \(\ln 25\) for two orders of 12.
2. **Product filter**: keep products bought by ≥ `min_customers` distinct customers.
3. **Drop customers** left with no rows.
4. **Values**: `log` = `log1p(total_qty)` (main model), `binary` = 1.0, `raw` = `total_qty` (both for the ML7 sensitivity check).
5. **Index maps**: sorted customer IDs → rows `0..n-1` (the anonymised customer index for R4), sorted StockCodes → cols `0..m-1`.
6. **csr_matrix** with `nnz` asserted equal to the number of aggregated pairs; all three matrices share shape and sparsity pattern.
7. **Flag test targets** `in_matrix` and compute the hit-rate ceiling.
8. **Save** matrices and index maps to `data/processed/` (gitignored); DE5 report to `reports/matrix_report.csv`.

In [1]:
# Environment guard: this notebook must run in the "giftlane" conda env (pandas 3.0.6).
# It fails fast with "Wrong Python environment. Run: conda activate giftlane" otherwise.
import sys
from pathlib import Path

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.env_check import check_env

print("environment OK:", check_env())

environment OK: /opt/homebrew/Caskroom/miniforge/base/envs/giftlane/bin/python


## 0. Setup — load TRAIN only

In [2]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.build_matrix import build_matrix, flag_targets, matrix_report, save_outputs

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

PROCESSED = ROOT / "data" / "processed"
train = pd.read_parquet(PROCESSED / "train.parquet")

print("python :", sys.executable)
print("pandas :", pd.__version__, "| scipy :", __import__("scipy").__version__)
print(f"train rows: {len(train):,} | customers: {train['Customer ID'].nunique():,} | "
      f"products: {train['StockCode'].nunique():,}")
print(f"train date range: {train['InvoiceDate'].min()} -> {train['InvoiceDate'].max()}")

python : /opt/homebrew/Caskroom/miniforge/base/envs/giftlane/bin/python
pandas : 3.0.6 | scipy : 1.17.1
train rows: 659,830 | customers: 5,472 | products: 4,531
train date range: 2009-12-01 07:45:00 -> 2011-10-07 16:06:00


## 1–6. Build the matrices

`build_matrix` runs steps 1–6 and records a one-line summary per step. `value="log"` selects the main matrix
(`result.A`); all three are in `result.matrices`.

In [3]:
MIN_CUSTOMERS = 10
result = build_matrix(train, value="log", min_customers=MIN_CUSTOMERS)
print("\n".join(result.steps))

[1] aggregate      : 659,830 train lines -> 417,501 (customer, product) pairs; 5,472 customers x 4,531 products
[2] product filter : keep products with >= 10 customers: 4,531 -> 3,687 products (844 dropped); pairs 417,501 -> 413,706 (3,795 removed)
[3] empty customers: 5,472 -> 5,471 customers (1 dropped, every product they bought fell under the filter)
[5] index maps     : 5,471 customers -> rows 0..5470; 3,687 products -> cols 0..3686
[4/6] matrices     : built log/binary/raw as csr float64, shape 5,471 x 3,687, nnz 413,706 == 413,706 pairs; identical shape and sparsity pattern


### Step 1 check on real data: aggregation happens before the transform

Pick a (customer, product) pair with several invoice lines in train and compare the stored `A_log` cell with
`log1p(sum qty)` (correct) and `sum(log1p(qty))` (the per-line mistake).

In [4]:
lines_per_pair = train.groupby(["Customer ID", "StockCode"]).size()
cust, prod = lines_per_pair.idxmax()
sub = train.loc[(train["Customer ID"] == cust) & (train["StockCode"] == prod), ["Invoice", "InvoiceDate", "Quantity"]]
row = int(result.customer_index.loc[result.customer_index["Customer ID"] == cust, "row"].iloc[0])
col = int(result.product_index.loc[result.product_index["StockCode"] == prod, "col"].iloc[0])

print(f"customer {cust}, product {prod}: {len(sub)} invoice lines, total qty {sub['Quantity'].sum():,}")
print(f"  A_raw[row, col]            = {result.matrices['raw'][row, col]:,.0f}")
print(f"  A_log[row, col]            = {result.matrices['log'][row, col]:.4f}")
print(f"  log1p(sum qty)  (correct)  = {np.log1p(sub['Quantity'].sum()):.4f}")
print(f"  sum(log1p(qty)) (wrong)    = {np.log1p(sub['Quantity']).sum():.4f}")
print(f"  A_binary[row, col]         = {result.matrices['binary'][row, col]:.0f}")
assert np.isclose(result.matrices["log"][row, col], np.log1p(sub["Quantity"].sum()))

customer 17850, product 82494L: 112 invoice lines, total qty 906
  A_raw[row, col]            = 906
  A_log[row, col]            = 6.8101
  log1p(sum qty)  (correct)  = 6.8101
  sum(log1p(qty)) (wrong)    = 242.3543
  A_binary[row, col]         = 1


### Steps 2–3 report: product filter and dropped customers

Products are counted by rows in the aggregated table (= distinct customers, since pairs are unique). Customers who
only bought filtered-out products have no row left. We check how many of those are **evaluation customers** —
this is the first time the test targets are read, and only to count overlap.

In [5]:
s = result.stats
print(f"min_customers = {s['min_customers']}")
print(f"products   : {s['products_before']:,} -> {s['products_after']:,}  ({s['products_dropped']:,} dropped)")
print(f"pairs      : {s['pairs_aggregated']:,} -> {s['pairs_after_product_filter']:,}  "
      f"({s['pairs_removed_by_product_filter']:,} removed, "
      f"{s['pairs_removed_by_product_filter'] / s['pairs_aggregated']:.2%} of pairs)")
print(f"customers  : {s['customers_before']:,} -> {s['customers_after']:,}  ({s['customers_dropped']:,} dropped)")

targets = pd.read_parquet(PROCESSED / "test_targets.parquet")[["Customer ID", "StockCode", "seen_in_train"]]
all_train_customers = train["Customer ID"].unique()
dropped_customers = np.setdiff1d(all_train_customers, result.customer_index["Customer ID"].to_numpy())
eval_customers = targets["Customer ID"].unique()
dropped_eval = np.intersect1d(dropped_customers, eval_customers)
print(f"dropped customers who are evaluation customers: {len(dropped_eval):,} of {len(eval_customers):,}")
if len(dropped_customers):
    print("dropped customer IDs:", dropped_customers.tolist())
    print(train.loc[train["Customer ID"].isin(dropped_customers), ["Customer ID", "StockCode", "Description", "Quantity"]]
          .drop_duplicates().to_string(index=False))

min_customers = 10
products   : 4,531 -> 3,687  (844 dropped)
pairs      : 417,501 -> 413,706  (3,795 removed, 0.91% of pairs)
customers  : 5,472 -> 5,471  (1 dropped)
dropped customers who are evaluation customers: 0 of 1,931
dropped customer IDs: [12503]
 Customer ID StockCode                      Description  Quantity
       12503    84760L LARGE HANGING GLASS+ZINC LANTERN       560


### Step 5: index maps

Row number is the anonymised customer index. `product_index` carries the most common train description and the
number of customers per product (always ≥ `min_customers`).

In [6]:
print("customer_index:", result.customer_index.shape, "| dtypes:", result.customer_index.dtypes.to_dict())
display(result.customer_index.head())
print("product_index:", result.product_index.shape, "| dtypes:", result.product_index.dtypes.to_dict())
display(result.product_index.head())
assert (result.customer_index["row"].to_numpy() == np.arange(len(result.customer_index))).all()
assert (result.product_index["col"].to_numpy() == np.arange(len(result.product_index))).all()
assert result.customer_index["Customer ID"].is_monotonic_increasing
assert result.product_index["StockCode"].is_monotonic_increasing

customer_index: (5471, 2) | dtypes: {'Customer ID': dtype('int64'), 'row': dtype('int64')}


,Customer ID,row
0,12346,0
1,12347,1
2,12348,2
3,12349,3
4,12350,4


product_index: (3687, 4) | dtypes: {'StockCode': <StringDtype(na_value=nan)>, 'col': dtype('int64'), 'Description': <StringDtype(na_value=nan)>, 'n_customers': dtype('int64')}


,StockCode,col,Description,n_customers
0,10002,0,INFLATABLE POLITICAL GLOBE,164
1,10080,1,GROOVY CACTUS INFLATABLE,16
2,10120,2,DOGGY RUBBER,47
3,10123C,3,HEARTS WRAPPING TAPE,39
4,10123G,4,ARMY CAMO WRAPPING TAPE,12


### Step 6: matrix assertions

In [7]:
A_log, A_bin, A_raw = (result.matrices[k] for k in ("log", "binary", "raw"))
n_pairs = len(result.pairs)
for name, A in result.matrices.items():
    assert isinstance(A, sp.csr_matrix) and A.dtype == np.float64
    assert A.nnz == n_pairs, name
    assert A.shape == A_log.shape, name
    assert np.array_equal(A.indptr, A_log.indptr) and np.array_equal(A.indices, A_log.indices), name
    print(f"{name:>7}: shape {A.shape}, nnz {A.nnz:,}, dtype {A.dtype}, min {A.data.min():.4f}, max {A.data.max():.4f}")
assert np.allclose(A_log.data, np.log1p(A_raw.data))
assert np.all(A_bin.data == 1.0)
print(f"\nnnz == aggregated pairs ({n_pairs:,}): OK  ->  no duplicates were summed inside csr_matrix")
print("identical shape and sparsity pattern across log / binary / raw: OK")

    log: shape (5471, 3687), nnz 413,706, dtype float64, min 0.6931, max 11.2147
 binary: shape (5471, 3687), nnz 413,706, dtype float64, min 1.0000, max 1.0000
    raw: shape (5471, 3687), nnz 413,706, dtype float64, min 1.0000, max 74215.0000

nnz == aggregated pairs (413,706): OK  ->  no duplicates were summed inside csr_matrix
identical shape and sparsity pattern across log / binary / raw: OK


## 7. Flag test targets and the hit-rate ceiling

`in_matrix = (customer in customer_index) AND (product in product_index)`. A target that is not in the matrix
cannot be recommended by any model built on this matrix. The share of evaluation customers with at least one
`in_matrix` target is the ceiling on hit rate.

In [8]:
targets_flagged, t = flag_targets(targets, result)
print(f"total target pairs                                : {t['targets_total']:,}")
print(f"in_matrix                                         : {t['targets_in_matrix']:,}  ({t['targets_in_matrix_pct']:.2f}%)")
print(f"   not in matrix because customer filtered        : {t['targets_customer_not_in_matrix']:,}")
print(f"   not in matrix because product filtered/unseen  : {t['targets_product_not_in_matrix']:,}")
print(f"evaluation customers                              : {t['eval_customers']:,}")
print(f"   present in matrix                              : {t['eval_customers_in_matrix']:,}")
print(f"   with >= 1 in_matrix target                     : {t['eval_customers_with_in_matrix_target']:,}")
print(f"hit-rate ceiling (customers with a reachable target): {t['hit_rate_ceiling_pct']:.2f}%")
print()
print(pd.crosstab(targets_flagged["seen_in_train"], targets_flagged["in_matrix"], margins=True))
display(targets_flagged.head())

total target pairs                                : 49,647
in_matrix                                         : 44,922  (90.48%)
   not in matrix because customer filtered        : 0
   not in matrix because product filtered/unseen  : 4,725
evaluation customers                              : 1,931
   present in matrix                              : 1,931
   with >= 1 in_matrix target                     : 1,913
hit-rate ceiling (customers with a reachable target): 99.07%

in_matrix      False   True    All
seen_in_train                     
False           3670      0   3670
True            1055  44922  45977
All             4725  44922  49647


,Customer ID,StockCode,seen_in_train,in_matrix
0,12347,21265,True,True
1,12347,21636,True,True
2,12347,22131,True,True
3,12347,22252,True,True
4,12347,22561,True,True


## DE5 — Matrix report for `A_log`

Shape, sparsity, dense vs sparse memory, value distribution, and non-zeros per row (products per customer) and per
column (customers per product). Saved to `reports/matrix_report.csv`.

In [9]:
report = matrix_report(A_log, "A_log")
with pd.option_context("display.float_format", lambda x: f"{x:,.4f}"):
    display(report)
report_path = ROOT / "reports" / "matrix_report.csv"
report.to_csv(report_path, index=False)
print("saved ->", report_path.relative_to(ROOT))

nnz_row = np.diff(A_log.indptr)
nnz_col = np.bincount(A_log.indices, minlength=A_log.shape[1])
print("\nproducts per customer quantiles :", {q: float(np.quantile(nnz_row, q)) for q in (0.1, 0.25, 0.5, 0.75, 0.9, 0.99)})
print("customers per product quantiles :", {q: float(np.quantile(nnz_col, q)) for q in (0.1, 0.25, 0.5, 0.75, 0.9, 0.99)})

,metric,value
0,matrix,A_log
1,rows (customers),5471
2,cols (products),3687
3,nnz,413706
4,sparsity_pct,97.9491
5,dense_memory_mb,161.3726
6,sparse_memory_mb,4.9864
7,compression_ratio_dense_over_sparse,32.3628
8,value_min,0.6931
9,value_median,1.9459


saved -> reports/matrix_report.csv

products per customer quantiles : {0.1: 8.0, 0.25: 18.0, 0.5: 41.0, 0.75: 93.0, 0.9: 180.0, 0.99: 468.90000000000055}
customers per product quantiles : {0.1: 15.0, 0.25: 28.0, 0.5: 66.0, 0.75: 149.0, 0.9: 275.0, 0.99: 587.6999999999994}


## 8. Save artefacts and confirm they are gitignored

In [10]:
written = save_outputs(result, targets_flagged, ROOT)
for p in written:
    print(f"saved -> {p.relative_to(ROOT)}  ({p.stat().st_size / 1e6:.2f} MB)")

# round-trip
B = sp.load_npz(PROCESSED / "A_log.npz")
assert B.shape == A_log.shape and B.nnz == A_log.nnz and np.allclose(B.data, A_log.data)
ci = pd.read_parquet(PROCESSED / "customer_index.parquet")
pi = pd.read_parquet(PROCESSED / "product_index.parquet")
tt = pd.read_parquet(PROCESSED / "test_targets.parquet")
assert len(ci) == A_log.shape[0] and len(pi) == A_log.shape[1]
assert list(tt.columns) == ["Customer ID", "StockCode", "seen_in_train", "in_matrix"]
print(f"\nround-trip OK: A_log {B.shape} nnz {B.nnz:,}; customer_index {len(ci):,}; product_index {len(pi):,}; "
      f"test_targets {len(tt):,} rows, columns {list(tt.columns)}")

try:
    out = subprocess.run(
        ["git", "check-ignore", "-v", *[str(p.relative_to(ROOT)) for p in written]],
        cwd=ROOT, capture_output=True, text=True, check=True,
    )
    print("\ngit check-ignore:\n" + out.stdout.strip())
except Exception as e:
    print("git check-ignore could not be run:", e)

saved -> data/processed/A_log.npz  (1.05 MB)
saved -> data/processed/A_binary.npz  (0.63 MB)
saved -> data/processed/A_raw.npz  (1.00 MB)
saved -> data/processed/customer_index.parquet  (0.06 MB)
saved -> data/processed/product_index.parquet  (0.11 MB)
saved -> data/processed/test_targets.parquet  (0.12 MB)

round-trip OK: A_log (5471, 3687) nnz 413,706; customer_index 5,471; product_index 3,687; test_targets 49,647 rows, columns ['Customer ID', 'StockCode', 'seen_in_train', 'in_matrix']

git check-ignore:
.gitignore:3:data/*	data/processed/A_log.npz
.gitignore:3:data/*	data/processed/A_binary.npz
.gitignore:3:data/*	data/processed/A_raw.npz
.gitignore:3:data/*	data/processed/customer_index.parquet
.gitignore:3:data/*	data/processed/product_index.parquet
.gitignore:3:data/*	data/processed/test_targets.parquet


## Notes

- `A_log` is the main model input; `A_binary` and `A_raw` are built on the identical sparsity pattern so the ML7
  sensitivity check changes *only* the cell values.
- `min_customers=10` is fitted on train only. The product filter removes < 1% of pairs but ~19% of products.
- Reasons and counts are recorded in `decision_log.md`.